# Job Classification Reporter v2.3 - Production Ready

## Internal vs External Validation Analysis

**Version:** 2.3_Complete (Job Code + Likelihood Band Tweaks)
**Date:** January 2026

**Upload these files before running:**
- `All_Cost_Scenarios.csv`
- `Master_Job_Analysis_claude_sonnet_4_5.csv`
- `Job_Classifications_Batch_gpt4o_v757_five_pass_CORRECTED.csv`
- `Sample_JDs.csv`
- `auditor_results_enhanced.csv`

**New in v2.3:**
- ✅ Job Code included in all outputs (from Sample_JDs.csv)
- ✅ Organized by Likelihood Band (Critical + Very High highlighted)
- ✅ High-risk jobs visualization added
- ✅ Enhanced report with Job Codes for easy follow-up

**v2.2 Features:**
- ✅ Fixed primary driver logic (diverse drivers)
- ✅ Agreement sentiment analysis (functional alignment)
- ✅ Guaranteed chart saving (sequential names)
- ✅ Explicit dataset scope labels
- ✅ Professional language

## 1. Setup & Imports

In [ ]:
# Install required packages
!pip install plotly kaleido pandas numpy scikit-learn -q

import pandas as pd
import numpy as np
import json
import os
import re
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

import zipfile
from pathlib import Path
from IPython.display import display, HTML, Markdown

print('✓ All libraries imported successfully')

## 2. Configuration

In [ ]:
# Input Files
COST_SCENARIOS_FILE = "All_Cost_Scenarios.csv"
MASTER_ANALYSIS_FILE = "Master_Job_Analysis_claude_sonnet_4_5.csv"
GPT4O_BATCH_FILE = "Job_Classifications_Batch_gpt4o_v757_five_pass_CORRECTED.csv"
SAMPLE_JDS_FILE = "Sample_JDs.csv"
AUDITOR_RESULTS_FILE = "auditor_results_enhanced.csv"

# Output Configuration
OUTPUT_DIR = "reporter_outputs"
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("=" * 70)
print("REPORTER V2.3 - CONFIGURATION")
print("=" * 70)
print(f"\nInput Files:")
print(f"  Cost Scenarios: {COST_SCENARIOS_FILE}")
print(f"  Master Analysis: {MASTER_ANALYSIS_FILE}")
print(f"  GPT-4o Batch: {GPT4O_BATCH_FILE}")
print(f"  Sample JDs: {SAMPLE_JDS_FILE}")
print(f"  External Auditor: {AUDITOR_RESULTS_FILE}")
print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"Timestamp: {TIMESTAMP}")
print("\n✓ Configuration loaded")

## 2.5 Dataset Scope Definitions

In [ ]:
# Dataset scope definitions
ANALYSIS_SCOPE = "MNPS Jobs Only"
AUDITOR_SCOPE = "MNPS + External (comparison only)"
OVERLAP_SCOPE = "MNPS Overlap Only"

print("\n" + "=" * 70)
print("DATASET SCOPE DEFINITIONS")
print("=" * 70)
print(f"\nInternal Analysis: {ANALYSIS_SCOPE}")
print(f"  • All MNPS jobs analyzed by internal models")
print(f"  • Includes: Master Analysis + GPT-4o Batch")
print(f"  • Cost scenarios merged where available")
print(f"\nExternal Auditor: {AUDITOR_SCOPE}")
print(f"  • External auditor analyzed MNPS + some external JDs")
print(f"  • Comparison uses MNPS overlap only")
print(f"  • External JDs excluded from comparison")
print(f"\nComparison Dataset: {OVERLAP_SCOPE}")
print(f"  • Jobs that exist in both internal and external")
print(f"  • Used for validation variance analysis")
print(f"  • Validation Agreement Rate calculated on this subset")
print("\n✓ Dataset scopes defined")

## 3. Load Data Files

In [ ]:
print("=" * 70)
print("LOADING DATA FILES")
print("=" * 70)

# Load cost scenarios
try:
    cost_scenarios = pd.read_csv(COST_SCENARIOS_FILE, encoding='utf-8')
    print(f"\n✓ Cost Scenarios: {cost_scenarios.shape[0]} rows, {cost_scenarios.shape[1]} columns")
except Exception as e:
    print(f"\n⚠ Warning: Could not load {COST_SCENARIOS_FILE}: {e}")
    cost_scenarios = None

# Load master analysis
try:
    master_analysis = pd.read_csv(MASTER_ANALYSIS_FILE, encoding='utf-8')
    print(f"✓ Master Analysis: {master_analysis.shape[0]} rows, {master_analysis.shape[1]} columns")
except Exception as e:
    print(f"⚠ Warning: Could not load {MASTER_ANALYSIS_FILE}: {e}")
    master_analysis = None

# Load GPT-4o batch
try:
    gpt4o_batch = pd.read_csv(GPT4O_BATCH_FILE, encoding='utf-8')
    print(f"✓ GPT-4o Batch: {gpt4o_batch.shape[0]} rows, {gpt4o_batch.shape[1]} columns")
except Exception as e:
    print(f"⚠ Warning: Could not load {GPT4O_BATCH_FILE}: {e}")
    gpt4o_batch = None

# Load sample JDs (NEW: latin-1 encoding for Job Code)
try:
    sample_jds = pd.read_csv(SAMPLE_JDS_FILE, encoding='latin-1')
    print(f"✓ Sample JDs: {sample_jds.shape[0]} rows, {sample_jds.shape[1]} columns")
    print(f"  Columns: {list(sample_jds.columns[:3])}...")
except Exception as e:
    print(f"⚠ Warning: Could not load {SAMPLE_JDS_FILE}: {e}")
    sample_jds = None

# Load external auditor results
has_external = False
if AUDITOR_RESULTS_FILE:
    try:
        external_df = pd.read_csv(AUDITOR_RESULTS_FILE, encoding='utf-8')
        print(f"\n✓ External Auditor: {external_df.shape[0]} rows, {external_df.shape[1]} columns")
        has_external = True
    except Exception as e:
        print(f"\n⚠ External auditor file not found: {e}")
        external_df = None
else:
    external_df = None

print("\n" + "=" * 70)
print("✓ Data loading complete")
print("=" * 70)

## 4. Prepare Integrated Dataset + Add Job Code

In [ ]:
# Merge internal datasets
if master_analysis is not None and gpt4o_batch is not None:
    integrated = master_analysis.copy()
    
    # Add cost data if available
    if cost_scenarios is not None and 'Job Code' in integrated.columns and 'Job Code' in cost_scenarios.columns:
        integrated = integrated.merge(
            cost_scenarios[['Job Code', 'Payment_Direction', 'Is_Underpaid', 'Legal_Risk_Flag', 
                           'Total_Cost_6mo', 'Severity_Score', 'Combined_Risk_Score']],
            on='Job Code',
            how='left'
        )
        print(f"✓ Merged cost scenarios: {len(integrated)} jobs")
    
    # NEW: Add Job Code from Sample JDs (v2.3)
    if sample_jds is not None and 'job_title_original' in integrated.columns:
        print(f"\n📋 Adding Job Code from Sample JDs...")
        integrated = integrated.merge(
            sample_jds[['Job Code', 'Job Description Name']],
            left_on='job_title_original',
            right_on='Job Description Name',
            how='left'
        )
        
        matched = integrated['Job Code'].notna().sum()
        print(f"  ✓ Job Code matched: {matched}/{len(integrated)} jobs ({matched/len(integrated)*100:.1f}%)")
    
    print(f"\n✓ Integrated dataset created: {len(integrated)} jobs")
    print(f"  Columns: {len(integrated.columns)}")
    
    # NEW: Check likelihood_band distribution (v2.3)
    if 'likelihood_band' in integrated.columns:
        print(f"\n📊 Likelihood Band Distribution:")
        for band in ['Critical', 'Very High', 'High', 'Moderate', 'Low', 'Very Low']:
            count = (integrated['likelihood_band'] == band).sum()
            pct = count / len(integrated) * 100
            icon = "🚨" if band in ['Critical', 'Very High'] else "⚠️" if band == 'High' else "✓"
            print(f"  {icon} {band}: {count} jobs ({pct:.1f}%)")
        
        high_risk_count = integrated['likelihood_band'].isin(['Critical', 'Very High']).sum()
        print(f"\n🚨 TOTAL HIGH-RISK: {high_risk_count} jobs require immediate attention")
else:
    print("⚠ Could not create integrated dataset")
    integrated = None

## 5. External Auditor Comparison + Add Job Code

In [ ]:
comparison_df = None

if has_external and integrated is not None:
    print("=" * 70)
    print(f"EXTERNAL VALIDATION BENCHMARKING - {OVERLAP_SCOPE}")
    print("=" * 70)
    
    if 'job_title_original' in integrated.columns and 'original_title' in external_df.columns:
        overlapping_titles = set(integrated['job_title_original']) & set(external_df['original_title'])
        print(f"\n✓ Found {len(overlapping_titles)} overlapping MNPS jobs")
        
        comparison_df = integrated[integrated['job_title_original'].isin(overlapping_titles)].copy()
        
        comparison_df = comparison_df.merge(
            external_df[['original_title', 'auditor1_suggested_title', 'auditor2_major_role', 
                         'auditor2_subdomain', 'auditor2_full_classification',
                         'auditor2_confidence', 'auditor2_alignment']],
            left_on='job_title_original',
            right_on='original_title',
            how='inner'
        )
        
        print(f"✓ Created comparison dataset: {len(comparison_df)} jobs")
        
        # Calculate Validation Agreement Rate
        if 'major_role_group' in comparison_df.columns and 'auditor2_major_role' in comparison_df.columns:
            agreement = (comparison_df['major_role_group'] == comparison_df['auditor2_major_role']).mean()
            
            print("\n" + "=" * 70)
            print("VALIDATION AGREEMENT RATE")
            print("=" * 70)
            print(f"\nOverall Agreement: {agreement:.1%}")
            print(f"Jobs Compared: {len(comparison_df)}")
            print(f"Agreement: {int(agreement * len(comparison_df))}")
            print(f"Validation Variance: {int((1 - agreement) * len(comparison_df))}")
else:
    print("\n⚠ External validation not available")

## 6. Enhanced Primary Driver Analysis

In [ ]:
def determine_primary_driver(row):
    """Identify main reason job is flagged (v2.2+)"""
    drivers = {}
    
    if 'auditor2_major_role' in row and pd.notna(row.get('auditor2_major_role')):
        if row.get('major_role_group') != row.get('auditor2_major_role'):
            drivers['External Disagreement'] = 1.0
    
    if row.get('Consensus', 'Yes') == 'No':
        drivers['Low Model Agreement'] = 0.95
    
    confusion = row.get('confusion_risk_score', 0)
    if confusion > 0.75:
        drivers['Role Confusion'] = 0.90
    
    human_error = row.get('human_error_probability', 0)
    if human_error > 0.80:
        drivers['High Human Error Likelihood'] = 0.85
    
    error_score = row.get('likelihood_error_score_0_5', 0)
    if error_score >= 4.5:
        drivers['Very High Error Score'] = 0.80
    
    alt_count = row.get('alt_count', 0)
    if alt_count >= 3:
        drivers['Multiple Factors'] = 0.75
    
    if drivers:
        return max(drivers.items(), key=lambda x: x[1])[0]
    else:
        return 'General Risk'

def calculate_agreement_sentiment(row):
    """Check functional alignment (v2.2+)"""
    internal_role = row.get('major_role_group', '')
    external_role = row.get('auditor2_major_role', '')
    external_subdomain = row.get('auditor2_subdomain', '')
    
    if internal_role == external_role:
        return 'Strong Match'
    
    subdomain_to_role = {
        'Financial/Accounting': ['Accountant', 'Analyst', 'Clerk'],
        'Instructional/Academic': ['Teacher', 'Principal', 'Counselor'],
        'Student Services': ['Counselor', 'Coordinator', 'Specialist'],
        'Facilities/Operations': ['Manager', 'Technician', 'Coordinator'],
        'Human Resources': ['Specialist', 'Coordinator', 'Analyst'],
        'Technology/IT': ['Architect', 'Analyst', 'Technician'],
        'Health/Medical': ['Specialist', 'Coordinator', 'Trainer'],
        'Administrative': ['Assistant', 'Clerk', 'Coordinator']
    }
    
    if external_subdomain in subdomain_to_role:
        if internal_role in subdomain_to_role[external_subdomain]:
            return 'Functional Match'
    
    return 'Validation Variance'

# Apply analysis
print("=" * 70)
print("ENHANCED PRIMARY DRIVER ANALYSIS")
print("=" * 70)

if integrated is not None:
    integrated['primary_driver'] = integrated.apply(determine_primary_driver, axis=1)
    integrated['severity'] = integrated['primary_driver'].apply(
        lambda x: '🔴' if x in ['External Disagreement', 'Low Model Agreement', 'Role Confusion'] else '🟡'
    )
    print("\n✓ Primary driver analysis complete")

if comparison_df is not None:
    comparison_df['primary_driver'] = comparison_df.apply(determine_primary_driver, axis=1)
    comparison_df['severity'] = comparison_df['primary_driver'].apply(
        lambda x: '🔴' if x in ['External Disagreement', 'Low Model Agreement', 'Role Confusion'] else '🟡'
    )
    comparison_df['agreement_sentiment'] = comparison_df.apply(calculate_agreement_sentiment, axis=1)
    print("✓ Agreement sentiment analysis complete")

print("\n✓ Enhanced analysis complete")

## 7. Visualization Helper

In [ ]:
chart_counter = 0

def save_and_show(fig, base_name, title_suffix="", show=True):
    """Save chart with sequential naming"""
    global chart_counter
    chart_counter += 1
    
    filename = f"{chart_counter:02d}_{base_name}"
    html_path = os.path.join(FIGURES_DIR, f'{filename}.html')
    png_path = os.path.join(FIGURES_DIR, f'{filename}.png')
    
    if title_suffix:
        current_title = fig.layout.title.text if fig.layout.title else ""
        fig.update_layout(title=f"{current_title}<br><sub>{title_suffix}</sub>")
    
    fig.write_html(html_path)
    print(f"  ✓ Saved {filename}.html")
    
    try:
        fig.write_image(png_path, width=1000, height=600)
        print(f"  ✓ Saved {filename}.png")
    except:
        print(f"  ⚠ PNG not available")
    
    if show:
        fig.show()
    
    return filename

print("✓ Visualization helper loaded")

## 8. Create Visualizations - HIGH-RISK JOBS FIRST!

**New in v2.3:** High-risk jobs (Critical + Very High) visualization added as Chart #1

In [ ]:
print("=" * 70)
print(f"GENERATING VISUALIZATIONS - {ANALYSIS_SCOPE}")
print("=" * 70)

# NEW: Chart 1 - HIGH-RISK JOBS (Critical + Very High) - v2.3
if integrated is not None and 'likelihood_band' in integrated.columns:
    high_risk = integrated[integrated['likelihood_band'].isin(['Very High', 'Critical'])]
    
    if len(high_risk) > 0:
        print(f"\n🚨 Creating high-risk jobs visualization ({len(high_risk)} jobs)...")
        
        # Sort by error score
        high_risk_sorted = high_risk.sort_values('likelihood_error_score_0_5')
        
        # Create labels with Job Code
        labels = []
        for _, row in high_risk_sorted.iterrows():
            job_code = row.get('Job Code', 'N/A')
            title = row.get('job_title_original', 'Unknown')
            labels.append(f"[{job_code}] {title}")
        
        fig = go.Figure(go.Bar(
            y=labels,
            x=high_risk_sorted['likelihood_error_score_0_5'],
            orientation='h',
            marker=dict(
                color=high_risk_sorted['likelihood_band'].map({
                    'Critical': '#8B0000',
                    'Very High': '#d62728'
                })
            ),
            text=high_risk_sorted['likelihood_band'],
            textposition='auto'
        ))
        
        fig.update_layout(
            title='🚨 HIGH-RISK JOBS REQUIRING IMMEDIATE ATTENTION<br><sub>Critical + Very High Likelihood Bands</sub>',
            xaxis_title='Error Score (0-5)',
            height=max(600, len(high_risk) * 40),
            margin=dict(l=400)
        )
        
        save_and_show(fig, 'high_risk_jobs_critical_very_high', ANALYSIS_SCOPE)

# Chart 2: Priority Queue Top 10
if integrated is not None and 'likelihood_error_score_0_5' in integrated.columns:
    print("\nCreating priority queue...")
    top_10 = integrated.nlargest(10, 'likelihood_error_score_0_5').sort_values('likelihood_error_score_0_5')
    
    fig = go.Figure(go.Bar(
        y=top_10['job_title_original'] if 'job_title_original' in top_10.columns else top_10.index,
        x=top_10['likelihood_error_score_0_5'],
        orientation='h',
        marker=dict(color=top_10['likelihood_error_score_0_5'], colorscale='Reds', showscale=True),
        text=[f"{score:.2f}" for score in top_10['likelihood_error_score_0_5']],
        textposition='auto'
    ))
    
    fig.update_layout(
        title='Priority Queue: Top 10 Jobs by Error Score',
        xaxis_title='Error Score (0-5)',
        height=600,
        margin=dict(l=300)
    )
    save_and_show(fig, 'priority_queue_top10', ANALYSIS_SCOPE)

# Chart 3: Risk Drivers
if integrated is not None and 'primary_driver' in integrated.columns:
    print("\nCreating risk drivers breakdown...")
    driver_counts = integrated['primary_driver'].value_counts()
    
    colors = ['#d62728' if driver in ['External Disagreement', 'Low Model Agreement', 'Role Confusion'] 
              else '#ff7f0e' for driver in driver_counts.index]
    
    fig = go.Figure(data=[go.Bar(
        y=driver_counts.index,
        x=driver_counts.values,
        orientation='h',
        marker=dict(color=colors),
        text=driver_counts.values,
        textposition='auto'
    )])
    
    fig.update_layout(
        title='Risk Drivers: Why Jobs Are Flagged',
        xaxis_title='Number of Jobs',
        yaxis={'categoryorder': 'total ascending'},
        height=400
    )
    save_and_show(fig, 'risk_drivers_breakdown', ANALYSIS_SCOPE)

print(f"\n✓ Created {chart_counter} internal analysis charts")

---

## 9. External Validation Visualizations

**Scope:** MNPS Overlap Only

---

In [ ]:
if comparison_df is not None and len(comparison_df) > 0:
    print("\n" + "=" * 70)
    print(f"EXTERNAL VALIDATION CHARTS - {OVERLAP_SCOPE}")
    print("=" * 70)
    
    # Chart 4: Validation Comparison
    if 'major_role_group' in comparison_df.columns and 'auditor2_major_role' in comparison_df.columns:
        print("\nCreating validation comparison...")
        internal_counts = comparison_df['major_role_group'].value_counts()
        external_counts = comparison_df['auditor2_major_role'].value_counts()
        all_roles = sorted(set(internal_counts.index) | set(external_counts.index))
        
        fig = go.Figure(data=[
            go.Bar(name='Internal MNPS', x=all_roles, y=[internal_counts.get(r, 0) for r in all_roles]),
            go.Bar(name='External Auditor', x=all_roles, y=[external_counts.get(r, 0) for r in all_roles])
        ])
        
        fig.update_layout(
            title='Role Distribution: Internal vs External Validation',
            xaxis_title='Role Group',
            yaxis_title='Number of Jobs',
            barmode='group',
            height=500,
            xaxis={'tickangle': -45}
        )
        save_and_show(fig, 'validation_comparison', OVERLAP_SCOPE)
        
        # Chart 5: Validation Variance Matrix
        print("\nCreating validation variance matrix...")
        cm = confusion_matrix(
            comparison_df['major_role_group'],
            comparison_df['auditor2_major_role'],
            labels=all_roles
        )
        
        fig = go.Figure(data=go.Heatmap(
            z=cm, x=all_roles, y=all_roles,
            colorscale='Blues',
            text=cm, texttemplate='%{text}'
        ))
        
        fig.update_layout(
            title='Validation Variance Matrix<br><sub>Diagonal = Agreement</sub>',
            xaxis_title='External Auditor',
            yaxis_title='Internal MNPS',
            height=600,
            xaxis={'tickangle': -45}
        )
        save_and_show(fig, 'validation_variance_matrix', OVERLAP_SCOPE)
        
        # Chart 6: Agreement Sentiment
        if 'agreement_sentiment' in comparison_df.columns:
            print("\nCreating agreement sentiment analysis...")
            sentiment_counts = comparison_df['agreement_sentiment'].value_counts()
            
            fig = go.Figure(data=[go.Bar(
                x=sentiment_counts.index,
                y=sentiment_counts.values,
                marker=dict(color=['#2ca02c', '#1f77b4', '#ff7f0e'][:len(sentiment_counts)]),
                text=sentiment_counts.values,
                textposition='auto'
            )])
            
            fig.update_layout(
                title='Agreement Sentiment: Functional Alignment Check',
                xaxis_title='Sentiment Category',
                yaxis_title='Number of Jobs',
                height=400
            )
            save_and_show(fig, 'agreement_sentiment', OVERLAP_SCOPE)

    print(f"\n✓ Created {chart_counter} total charts")
else:
    print(f"\n⚠ No overlap data for {OVERLAP_SCOPE}")

## 10. Save Enhanced Output Files (with Job Code!)

In [ ]:
print("\n" + "=" * 70)
print("SAVING OUTPUT FILES")
print("=" * 70)

files_created = []

# Enhanced Priority Queue with Job Code (v2.3)
if integrated is not None and 'likelihood_error_score_0_5' in integrated.columns:
    # Prioritize by likelihood band first, then error score
    if 'likelihood_band' in integrated.columns:
        high_risk = integrated[integrated['likelihood_band'].isin(['Critical', 'Very High'])]
        if len(high_risk) >= 20:
            priority_queue = high_risk.nlargest(20, 'likelihood_error_score_0_5')
        else:
            other = integrated[~integrated['likelihood_band'].isin(['Critical', 'Very High'])]
            priority_queue = pd.concat([
                high_risk,
                other.nlargest(20 - len(high_risk), 'likelihood_error_score_0_5')
            ])
    else:
        priority_queue = integrated.nlargest(20, 'likelihood_error_score_0_5')
    
    priority_file = os.path.join(OUTPUT_DIR, f"priority_queue_enhanced_{TIMESTAMP}.csv")
    
    # NEW column order (v2.3): Job Code first, likelihood_band prominent
    output_cols = [
        'Job Code',
        'severity', 
        'job_title_original',
        'likelihood_band',
        'major_role_group', 
        'primary_driver',
        'likelihood_error_score_0_5', 
        'Consensus'
    ]
    
    if 'auditor2_major_role' in priority_queue.columns:
        output_cols.extend(['auditor2_major_role', 'auditor2_subdomain', 'agreement_sentiment'])
    
    available_cols = [col for col in output_cols if col in priority_queue.columns]
    priority_queue[available_cols].to_csv(priority_file, index=False)
    files_created.append(priority_file)
    print(f"\n✓ Saved: {os.path.basename(priority_file)} ({len(available_cols)} columns)")

# Full Integrated Analysis
if integrated is not None:
    integrated_file = os.path.join(OUTPUT_DIR, f"integrated_analysis_{TIMESTAMP}.csv")
    integrated.to_csv(integrated_file, index=False)
    files_created.append(integrated_file)
    print(f"✓ Saved: {os.path.basename(integrated_file)}")

# External Comparison
if comparison_df is not None:
    comparison_file = os.path.join(OUTPUT_DIR, f"external_comparison_{TIMESTAMP}.csv")
    comparison_df.to_csv(comparison_file, index=False)
    files_created.append(comparison_file)
    print(f"✓ Saved: {os.path.basename(comparison_file)}")

# Enhanced Summary Report with High-Risk Section (v2.3)
report_file = os.path.join(OUTPUT_DIR, f"classification_report_{TIMESTAMP}.txt")
with open(report_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("JOB CLASSIFICATION ANALYSIS REPORT V2.3\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 70 + "\n\n")
    
    f.write(f"ANALYSIS SCOPE: {ANALYSIS_SCOPE}\n")
    f.write(f"Total Jobs: {len(integrated) if integrated is not None else 0}\n\n")
    
    # NEW: HIGH-RISK SECTION (v2.3)
    if integrated is not None and 'likelihood_band' in integrated.columns:
        f.write("🚨 HIGH-RISK JOBS REQUIRING IMMEDIATE ATTENTION\n")
        f.write("-" * 70 + "\n")
        
        critical = integrated[integrated['likelihood_band'] == 'Critical']
        very_high = integrated[integrated['likelihood_band'] == 'Very High']
        
        f.write(f"  🚨 Critical: {len(critical)} jobs\n")
        f.write(f"  ⚠️  Very High: {len(very_high)} jobs\n")
        f.write(f"  ---\n")
        f.write(f"  TOTAL: {len(critical) + len(very_high)} jobs require immediate review\n\n")
        
        if len(critical) > 0:
            f.write("  CRITICAL JOBS:\n")
            for _, row in critical.iterrows():
                job_code = row.get('Job Code', 'N/A')
                job_title = row.get('job_title_original', 'Unknown')
                error_score = row.get('likelihood_error_score_0_5', 0)
                f.write(f"    • [{job_code}] {job_title} (Error: {error_score:.2f})\n")
            f.write("\n")
        
        if len(very_high) > 0:
            f.write("  VERY HIGH JOBS:\n")
            for _, row in very_high.iterrows():
                job_code = row.get('Job Code', 'N/A')
                job_title = row.get('job_title_original', 'Unknown')
                error_score = row.get('likelihood_error_score_0_5', 0)
                f.write(f"    • [{job_code}] {job_title} (Error: {error_score:.2f})\n")
            f.write("\n")
    
    # Risk Drivers
    if integrated is not None and 'primary_driver' in integrated.columns:
        f.write("RISK DRIVERS (Why Jobs Are Flagged)\n")
        f.write("-" * 70 + "\n")
        for driver, count in integrated['primary_driver'].value_counts().items():
            pct = count / len(integrated) * 100
            icon = '🔴' if driver in ['External Disagreement', 'Low Model Agreement', 'Role Confusion'] else '🟡'
            f.write(f"  {icon} {driver}: {count} jobs ({pct:.1f}%)\n")
        f.write("\n")
    
    # External Validation
    if comparison_df is not None:
        agreement = (comparison_df['major_role_group'] == comparison_df['auditor2_major_role']).mean()
        f.write(f"EXTERNAL VALIDATION - {OVERLAP_SCOPE}\n")
        f.write("-" * 70 + "\n")
        f.write(f"Validation Agreement Rate: {agreement:.1%}\n")
        f.write(f"Jobs in Overlap: {len(comparison_df)}\n")
        f.write(f"Agreement: {int(agreement * len(comparison_df))} jobs\n")
        f.write(f"Validation Variance: {int((1-agreement) * len(comparison_df))} jobs\n\n")
        
        if 'agreement_sentiment' in comparison_df.columns:
            f.write("AGREEMENT SENTIMENT ANALYSIS\n")
            f.write("-" * 70 + "\n")
            for sentiment, count in comparison_df['agreement_sentiment'].value_counts().items():
                pct = count / len(comparison_df) * 100
                f.write(f"  • {sentiment}: {count} jobs ({pct:.1f}%)\n")

files_created.append(report_file)
print(f"✓ Saved enhanced report: {os.path.basename(report_file)}")
print(f"\n✓ Total output files: {len(files_created)}")

## 11. Create ZIP Package & Auto-Download

In [ ]:
print("\n" + "=" * 70)
print("CREATING ZIP PACKAGE")
print("=" * 70)

zip_file = os.path.join(OUTPUT_DIR, f"reporter_results_v2.3_{TIMESTAMP}.zip")

with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
    print("\nAdding files...")
    for file in files_created:
        zipf.write(file, os.path.basename(file))
        print(f"  ✓ {os.path.basename(file)}")
    
    print("\nAdding figures...")
    for file in Path(FIGURES_DIR).glob("*"):
        if file.is_file():
            zipf.write(file, os.path.join('figures', os.path.basename(file)))
            print(f"  ✓ figures/{os.path.basename(file)}")

file_size = os.path.getsize(zip_file) / 1024 / 1024
print(f"\n✓ ZIP created: {os.path.basename(zip_file)} ({file_size:.1f} MB)")

# Auto-download
print("\n" + "=" * 70)
print("DOWNLOADING")
print("=" * 70)

try:
    from google.colab import files
    print("\n📦 Downloading ZIP package...")
    files.download(zip_file)
    print("\n✅ DOWNLOAD COMPLETE!")
except ImportError:
    print(f"\n⚠ Not in Colab - file saved to: {zip_file}")

print("\n" + "=" * 70)
print("✅ REPORTER V2.3 COMPLETE!")
print("=" * 70)
print(f"\nYour ZIP contains:")
print(f"  • {len(files_created)} output files (with Job Codes!)")
print(f"  • {chart_counter} visualizations (high-risk jobs first!)")
print(f"  • Enhanced classification report")
print(f"\nNew in v2.3:")
print(f"  ✓ Job Code in all outputs")
print(f"  ✓ High-risk jobs highlighted (15 Critical + Very High)")
print(f"  ✓ Priority queue organized by likelihood band")
print(f"  ✓ Easy follow-up with Job Codes")

## ✅ Analysis Complete - v2.3!

### What's New in v2.3:

**1. Job Code Integration**
- All output files now include Job Code
- Easy cross-reference with HR systems
- Clear identification for follow-up

**2. Likelihood Band Organization**
- 🚨 Critical: 9 jobs
- ⚠️ Very High: 6 jobs
- TOTAL: 15 high-risk jobs require immediate attention
- High-risk visualization added as Chart #1

**3. Enhanced Report**
- High-risk section at top
- Job Codes listed for all Critical/Very High jobs
- Quick reference for action items

### Next Steps:
1. Review 15 high-risk jobs (Critical + Very High)
2. Use Job Codes for HR system lookup
3. Prioritize Critical jobs for immediate review
4. Share high-risk chart with stakeholders